# 01 · Data download, EDA and near-duplicate audit

**Run `00_colab_bootstrap` logic first? No need:** the first cells below repeat the short setup.
Select a **T4 GPU or CPU** runtime (this notebook does not need a GPU).

What it does

1. clones the repo, installs `plantdx`, points artifacts to Drive,
2. downloads `data.zip` (2.2 GB) and extracts only the colour images to `/content/data` (local disk, fast),
3. checks that every image listed in `splits/*.csv` exists,
4. makes the EDA figures,
5. hashes all images and measures how many val/test images have a near-identical same-class image in an earlier split, separately for leaf-verified (`map`) and unverified (`fallback`) images,
6. packs everything into one zip you download and put into the repo (`reports/`).

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
!git log -1 --oneline
!ls splits

## 2 · Download and extract the colour images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT = "color"
DATA_ROOT = "/content/data"

zip_path = download_archive()
print("archive:", zip_path)
print("new files extracted:", extract_variant(zip_path, DATA_ROOT, VARIANT))

## 3 · Load splits and check that every file exists

In [ ]:
import json
import pandas as pd

from plantdx.data.images import missing_files

classes = json.load(open("splits/classes.json", encoding="utf-8"))
table = pd.concat(
    [pd.read_csv(f"splits/{s}.csv").assign(split=s) for s in ("train", "val", "test")],
    ignore_index=True,
)
missing = missing_files(DATA_ROOT, VARIANT, table["path"])
print("images in splits:", len(table), "| missing on disk:", len(missing))
assert not missing, missing[:5]
print(json.dumps(json.load(open("splits/summary.json")), indent=2))

## 4 · EDA figures

In [ ]:
from pathlib import Path
from PIL import Image

from plantdx.viz import plot_class_counts, plot_fallback_share, sample_grid

FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
for p in (
    plot_class_counts(table, classes, FIG / "eda_class_counts.png"),
    plot_fallback_share(table, classes, FIG / "eda_fallback_share.png"),
    sample_grid(table, classes, DATA_ROOT, VARIANT, FIG / "eda_sample_grid.png"),
):
    display(Image.open(p))

# image size / mode statistics on a random sample
from collections import Counter
sample = table.sample(min(1000, len(table)), random_state=42)["path"]
sizes = Counter()
for rel in sample:
    with Image.open(f"{DATA_ROOT}/raw/{VARIANT}/{rel}") as im:
        sizes[(im.size, im.mode)] += 1
print("image (size, mode) in a 1000-image sample:", sizes.most_common(5))

print("\nimages per class (train): min", table[table.split == "train"].groupby("label_index").size().min(),
      "max", table[table.split == "train"].groupby("label_index").size().max())

## 5 · Near-duplicate audit

Hashes (256-bit dHash) are cached on Drive, so a restarted session does not recompute them.
For every val image we look for the nearest same-class image in train; for every test image,
in train + val. Distance 0 = identical hash; larger = less similar (max 256).

In [ ]:
import numpy as np

from plantdx.data.audit import hash_paths, nearest_cross_split, summarize_matches
from plantdx.data.images import image_path
from plantdx.paths import artifacts_dir

cache = artifacts_dir() / "hashes" / f"dhash256_{VARIANT}.npz"
cache.parent.mkdir(parents=True, exist_ok=True)

paths = [image_path(DATA_ROOT, VARIANT, p) for p in table["path"]]
path_list = table["path"].tolist()
hashes = None
if cache.exists():
    z = np.load(cache)
    if z["paths"].tolist() == path_list:
        hashes = z["hashes"]
        print("loaded cached hashes", hashes.shape)
if hashes is None:
    hashes = hash_paths(paths, workers=os.cpu_count() or 2)
    np.savez(cache, hashes=hashes, paths=np.array(path_list))
    print("computed and cached", hashes.shape)

In [ ]:
matches = pd.concat(
    [
        nearest_cross_split(table, hashes, "val", ("train",)),
        nearest_cross_split(table, hashes, "test", ("train", "val")),
    ],
    ignore_index=True,
)
summary = summarize_matches(matches)
print("Share (%) of images that have a same-class neighbour at Hamming distance <= t")
display(summary)

matches.sort_values("distance").to_csv("reports/audit_near_duplicates.csv", index=False)
summary.to_csv("reports/audit_summary.csv", index=False)

**How to read the table.** Rows `map` are the control: those images have verified leaf ids, so any
close match there is a *look-alike of another leaf*, not leakage. If `fallback` rows show clearly higher shares at
small distances (≤ 5, ≤ 10) than `map` rows, that excess is residual near-duplicate leakage. If they look alike,
the fallback images are about as clean as the verified ones.

In [ ]:
from plantdx.viz import plot_audit_pairs

fb = matches[matches.leaf_source == "fallback"]
display(Image.open(plot_audit_pairs(fb[fb.distance <= 20], DATA_ROOT, VARIANT, FIG / "audit_closest_pairs.png")))

## 6 · Pack the results

In [ ]:
import shutil
from google.colab import files

shutil.copytree("reports", artifacts_dir() / "reports_step2", dirs_exist_ok=True)  # backup on Drive
shutil.make_archive("/content/step2_reports", "zip", ".", "reports")
files.download("/content/step2_reports.zip")